# E3e Conservative ViT Fine-Tuning

Thin Colab launcher for the E3e conservative ViT fine-tuning diagnostic. Core logic stays in `src/` and `scripts/`. This notebook is validation-only: do not compute test metrics here.

Outputs are isolated under `finetuned_vit_last2_lr5e6`, `finetuned_vit_last1_lr5e6`, and `/content/drive/MyDrive/dl-final-artifact/e3e_conservative_vit/`.

## Runtime

Use `Runtime > Change runtime type` and select a GPU before running full fine-tuning.

In [ ]:
!nvidia-smi

## Drive and Repository Setup

This cell clones or fast-forwards the GitHub repo, then restores the HAM10000 bundle if it is available on Drive.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/dl-final-artifact"
E3E_DRIVE_ROOT = f"{DRIVE_ROOT}/e3e_conservative_vit"
REPO_URL = "https://github.com/KasimDeliaci/dl-final-project.git"
REPO_DIR = "/content/dl-final-project"

!mkdir -p "$DRIVE_ROOT" "$E3E_DRIVE_ROOT"
!if [ ! -d "$REPO_DIR/.git" ]; then git clone "$REPO_URL" "$REPO_DIR"; fi
%cd /content/dl-final-project
!git pull --ff-only
!git rev-parse --short HEAD

BUNDLE_CANDIDATES = [
    f"{DRIVE_ROOT}/ham10000_colab_bundle.tar",
    "/content/drive/MyDrive/ham10000_colab_bundle.tar",
    "/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar",
]
bundle = next((path for path in BUNDLE_CANDIDATES if Path(path).exists()), None)
print("HAM10000 bundle:", bundle or "not found")
if bundle:
    !tar -xf "$bundle" -C /content/dl-final-project
else:
    print(
        "Upload/copy ham10000_colab_bundle.tar before full runs, "
        "or ensure data/ paths already exist."
    )

## Install Dependencies

In [ ]:
!pip -q install uv
!uv sync --dev

## Canonical Split Repair and Check

This regenerates the lesion-aware split after restoring the data bundle and fails fast if row counts differ from the project protocol.

In [ ]:
!PYTHONPATH=src uv run python scripts/audit_ham10000.py \
  --config configs/dataset/selected_dataset.yaml
!PYTHONPATH=src uv run python scripts/make_lesion_split.py \
  --config configs/dataset/selected_dataset.yaml

import pandas as pd

EXPECTED_SPLIT_ROWS = {"train": 7008, "val": 1504, "test": 1503}
for split, expected in EXPECTED_SPLIT_ROWS.items():
    path = Path("data/splits") / f"{split}.csv"
    actual = len(pd.read_csv(path))
    print(split, actual)
    if actual != expected:
        raise RuntimeError(f"{split}.csv has {actual} rows; expected {expected}.")

## Restore Canonical Swin/BEiT Feature Caches

E3e only fine-tunes ViT. For mixed triple-fusion checks, this restores canonical Sprint 4 Swin and BEiT caches from Drive without modifying the canonical Drive source.

In [ ]:
from pathlib import Path

CANONICAL_DRIVE_FEATURES = Path(DRIVE_ROOT) / "artifacts/features/ham10000/finetuned"
LOCAL_CANONICAL_FEATURES = Path("artifacts/features/ham10000/finetuned")
for backbone in ["swin_tiny", "beit_base"]:
    source = CANONICAL_DRIVE_FEATURES / backbone
    dest = LOCAL_CANONICAL_FEATURES / backbone
    if not source.exists():
        raise FileNotFoundError(
            f"Missing canonical {backbone} cache on Drive: {source}. "
            "Sync Sprint 4 artifacts before running E3e fusion."
        )
    !mkdir -p "{dest.parent}"
    !rsync -a "{source}/" "{dest}/"
print("Canonical Swin/BEiT caches restored locally for mixed-source fusion.")

## Fast Verification

In [ ]:
!PYTHONPATH=src uv run ruff check .
!PYTHONPATH=src uv run python -m pytest \
  tests/test_dataset_sprint1.py \
  tests/test_sprint2_features.py \
  tests/test_sprint3_fusion.py \
  tests/test_sprint4_finetune.py

## Smoke Fine-Tuning Run

This checks the E3e code path without downloading pretrained weights or running the full dataset.

In [ ]:
!PYTHONPATH=src uv run python scripts/finetune_backbone.py \
  --config configs/experiments/e3e_vit_last1_lr5e6.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --backbones vit_b16 \
  --epochs 1 \
  --batch-size 1 \
  --num-workers 0 \
  --limit-per-split 2 \
  --no-pretrained \
  --no-mixed-precision \
  --checkpoint-dir artifacts/checkpoints_smoke/ham10000/e3e_vit_last1_lr5e6 \
  --feature-root artifacts/features_smoke \
  --run-root artifacts/runs_smoke

## Full E3e ViT Fine-Tuning

Runs the two planned conservative ViT conditions. Test metrics are intentionally not computed.

In [ ]:
!PYTHONUNBUFFERED=1 PYTHONPATH=src uv run python scripts/finetune_backbone.py \
  --config configs/experiments/e3e_vit_last2_lr5e6.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --backbones vit_b16 \
  --batch-size 16 \
  --num-workers 2

!PYTHONUNBUFFERED=1 PYTHONPATH=src uv run python scripts/finetune_backbone.py \
  --config configs/experiments/e3e_vit_last1_lr5e6.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --backbones vit_b16 \
  --batch-size 16 \
  --num-workers 2

## E3e ViT Single-Backbone MLP

In [ ]:
import os
import subprocess

env = {**os.environ, "PYTHONPATH": "src"}
for feature_source in ["finetuned_vit_last2_lr5e6", "finetuned_vit_last1_lr5e6"]:
    subprocess.run(
        [
            "uv",
            "run",
            "python",
            "scripts/train_mlp.py",
            "--dataset-config",
            "configs/dataset/selected_dataset.yaml",
            "--feature-source",
            feature_source,
            "--backbones",
            "vit_b16",
            "--batch-size",
            "128",
            "--experiment-id",
            "E3e",
            "--test-policy",
            "not_loaded_or_used_in_e3e",
            "--run-tag",
            "e3e_vit_single",
        ],
        check=True,
        env=env,
    )

## Build Mixed Feature Sources

These sources combine the new E3e ViT cache with canonical Sprint 4 Swin/BEiT caches for validation-only triple concat checks.

In [ ]:
import shutil

FEATURE_ROOT = Path("artifacts/features/ham10000")
MIXED_SOURCES = {
    "finetuned_vit_last2_lr5e6": "finetuned_vit_last2_lr5e6_plus_s4_swin_beit",
    "finetuned_vit_last1_lr5e6": "finetuned_vit_last1_lr5e6_plus_s4_swin_beit",
}
for vit_source, mixed_source in MIXED_SOURCES.items():
    mixed_root = FEATURE_ROOT / mixed_source
    shutil.rmtree(mixed_root, ignore_errors=True)
    for backbone, source_root in [
        ("vit_b16", FEATURE_ROOT / vit_source),
        ("swin_tiny", FEATURE_ROOT / "finetuned"),
        ("beit_base", FEATURE_ROOT / "finetuned"),
    ]:
        src = source_root / backbone
        dst = mixed_root / backbone
        if not src.exists():
            raise FileNotFoundError(f"Missing cache source: {src}")
        shutil.copytree(src, dst)
    print("Created", mixed_root)

## E3e Mixed Triple Concat Fusion

In [ ]:
for feature_source in [
    "finetuned_vit_last2_lr5e6_plus_s4_swin_beit",
    "finetuned_vit_last1_lr5e6_plus_s4_swin_beit",
]:
    subprocess.run(
        [
            "uv",
            "run",
            "python",
            "scripts/run_fusion_matrix.py",
            "--feature-source",
            feature_source,
            "--only-combination",
            "vit_b16",
            "swin_tiny",
            "beit_base",
            "--fusion-methods",
            "concat",
            "--batch-size",
            "128",
            "--experiment-id",
            "E3e",
            "--test-policy",
            "not_loaded_or_used_in_e3e",
            "--run-tag",
            "e3e_mixed_triple",
        ],
        check=True,
        env=env,
    )

## Optional Metadata-Conditioned Follow-Up

Run only if the single ViT or mixed concat result is competitive enough to justify repeating E3d operators. This remains validation-only.

In [ ]:
RUN_OPTIONAL_METADATA_OPERATORS = False

if RUN_OPTIONAL_METADATA_OPERATORS:
    for feature_source in [
        "finetuned_vit_last2_lr5e6_plus_s4_swin_beit",
        "finetuned_vit_last1_lr5e6_plus_s4_swin_beit",
    ]:
        subprocess.run(
            [
                "uv",
                "run",
                "python",
                "scripts/train_metadata_fusion_operator.py",
                "--feature-source",
                feature_source,
                "--operators",
                "triple_metadata_film",
                "triple_metadata_gated_backbone",
                "--device",
                "cpu",
                "--batch-size",
                "128",
                "--experiment-id",
                "E3e",
                "--test-policy",
                "not_loaded_or_used_in_e3e",
                "--run-tag",
                "e3e_metadata_followup",
            ],
            check=True,
            env=env,
        )

## Artifact Integrity Checks

In [ ]:
import json

import torch

EXPECTED_CACHE_ROWS = {"train": 7008, "val": 1504}
sources_to_check = [
    "finetuned_vit_last2_lr5e6",
    "finetuned_vit_last1_lr5e6",
    "finetuned_vit_last2_lr5e6_plus_s4_swin_beit",
    "finetuned_vit_last1_lr5e6_plus_s4_swin_beit",
]
rows = []
for source in sources_to_check:
    source_root = Path("artifacts/features/ham10000") / source
    backbones = ["vit_b16"] if "plus_s4" not in source else ["vit_b16", "swin_tiny", "beit_base"]
    for backbone in backbones:
        for split, expected in EXPECTED_CACHE_ROWS.items():
            path = source_root / backbone / f"{split}.pt"
            if not path.exists():
                rows.append(
                    {
                        "source": source,
                        "backbone": backbone,
                        "split": split,
                        "status": "missing",
                        "path": str(path),
                    }
                )
                continue
            payload = torch.load(path, map_location="cpu", weights_only=False)
            features = payload["features"]
            metadata = payload["metadata"]
            rows.append({
                "source": source,
                "backbone": backbone,
                "split": split,
                "rows": int(features.shape[0]),
                "expected_rows": expected,
                "feature_dim": int(features.shape[1]),
                "finite": bool(torch.isfinite(features).all()),
                "metadata_feature_source": metadata.get("feature_source"),
                "selection_metric": metadata.get("config", {}).get("selection_metric"),
                "test_policy": metadata.get("config", {}).get("test_policy"),
                "path": str(path),
            })
cache_checks = pd.DataFrame(rows)
display(cache_checks)
default_false = pd.Series(False, index=cache_checks.index)
default_zero = pd.Series(0, index=cache_checks.index)
missing = cache_checks.get("status", default_false) == "missing"
row_mismatch = cache_checks.get("rows", default_zero) != cache_checks.get(
    "expected_rows",
    default_zero,
)
non_finite = ~cache_checks.get("finite", default_false).fillna(False).astype(bool)
if len(cache_checks[missing | row_mismatch | non_finite]):
    raise RuntimeError("E3e cache checks need attention before interpreting results.")
print("E3e cache checks passed.")

prediction_rows = []
for path in sorted(Path("artifacts/runs").glob("*/predictions.csv")):
    config_path = path.parent / "run_config.json"
    if not config_path.exists():
        continue
    config = json.loads(config_path.read_text())
    if config.get("experiment_id") != "E3e":
        continue
    frame = pd.read_csv(path)
    prediction_rows.append({
        "run_id": config.get("run_id"),
        "feature_source": config.get("feature_source"),
        "fusion_method": config.get("fusion_method"),
        "backbone": config.get("backbone"),
        "rows": len(frame),
        "prob_cols": len([column for column in frame.columns if column.startswith("prob_")]),
        "test_policy": config.get("test_policy"),
        "path": str(path),
    })
prediction_checks = pd.DataFrame(prediction_rows)
display(prediction_checks.sort_values(["feature_source", "fusion_method", "backbone"]))
if len(prediction_checks):
    valid_predictions = (prediction_checks["rows"] == 1504) & (
        prediction_checks["prob_cols"] == 7
    )
else:
    valid_predictions = pd.Series(True, dtype=bool)
if not valid_predictions.all():
    raise RuntimeError("E3e prediction dump checks failed.")

## Training Curves, Confusion Matrices, and Tables

In [ ]:
from IPython.display import Image, Markdown, display

RUN_ROOT = Path("artifacts/runs")
TABLE_ROOT = Path("artifacts/report_assets/tables")
FIGURE_ROOT = Path("artifacts/report_assets/figures")

for run_dir in sorted(RUN_ROOT.glob("*e3e*")):
    metrics_path = run_dir / "metrics_summary.csv"
    if metrics_path.exists():
        display(Markdown(f"### {run_dir.name}"))
        display(pd.read_csv(metrics_path))
    for figure_name in ["training_curves.png", "confusion_matrix.png"]:
        figure_path = run_dir / figure_name
        if figure_path.exists():
            display(Image(str(figure_path)))

for path in sorted(TABLE_ROOT.glob("*finetuned_vit_last*")):
    display(Markdown(f"### {path}"))
    if path.stat().st_size == 0:
        display(Markdown("Empty table file. No rows were exported for this artifact."))
        continue
    try:
        display(pd.read_csv(path))
    except pd.errors.EmptyDataError:
        display(Markdown("Empty table file. No rows were exported for this artifact."))
for path in sorted(FIGURE_ROOT.glob("*finetuned_vit_last*")):
    display(Markdown(f"### {path}"))
    display(Image(str(path)))

## Sync E3e Artifacts to Drive

Generated checkpoints, features, runs, predictions, and figures are intentionally not committed to Git.

In [ ]:
import subprocess
from pathlib import Path

drive_artifact_root = Path(E3E_DRIVE_ROOT) / "artifacts"


def sync_directory(source: str, dest: str) -> None:
    source_path = Path(source)
    dest_path = Path(dest)
    if not source_path.exists():
        print(f"WARNING: missing source, skipping sync: {source_path}")
        return
    dest_path.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-av", f"{source_path}/", f"{dest_path}/"], check=True)


def sync_matching_dirs(pattern: str, source: str, dest: str) -> None:
    source_path = Path(source)
    dest_path = Path(dest)
    if not source_path.exists():
        print(f"WARNING: missing source, skipping sync: {source_path}")
        return
    dest_path.mkdir(parents=True, exist_ok=True)
    matches = [path for path in sorted(source_path.glob(pattern)) if path.is_dir()]
    if not matches:
        print(f"WARNING: no directories matched {pattern} under {source_path}")
    for path in matches:
        target = dest_path / path.name
        target.mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-av", f"{path}/", f"{target}/"], check=True)


def sync_matching_files(pattern: str, source: str, dest: str) -> None:
    source_path = Path(source)
    dest_path = Path(dest)
    if not source_path.exists():
        print(f"WARNING: missing source, skipping sync: {source_path}")
        return
    dest_path.mkdir(parents=True, exist_ok=True)
    matches = [path for path in sorted(source_path.glob(pattern)) if path.is_file()]
    if not matches:
        print(f"WARNING: no files matched {pattern} under {source_path}")
    for path in matches:
        subprocess.run(["rsync", "-av", str(path), f"{dest_path}/"], check=True)

for relative in [
    "runs",
    "report_assets/tables",
    "report_assets/figures",
]:
    (drive_artifact_root / relative).mkdir(parents=True, exist_ok=True)

sync_pairs = [
    (
        "artifacts/checkpoints/ham10000/e3e_vit_last2_lr5e6/",
        str(drive_artifact_root / "checkpoints/ham10000/e3e_vit_last2_lr5e6/"),
    ),
    (
        "artifacts/checkpoints/ham10000/e3e_vit_last1_lr5e6/",
        str(drive_artifact_root / "checkpoints/ham10000/e3e_vit_last1_lr5e6/"),
    ),
    (
        "artifacts/features/ham10000/finetuned_vit_last2_lr5e6/",
        str(drive_artifact_root / "features/ham10000/finetuned_vit_last2_lr5e6/"),
    ),
    (
        "artifacts/features/ham10000/finetuned_vit_last1_lr5e6/",
        str(drive_artifact_root / "features/ham10000/finetuned_vit_last1_lr5e6/"),
    ),
    (
        "artifacts/features/ham10000/finetuned_vit_last2_lr5e6_plus_s4_swin_beit/",
        str(
            drive_artifact_root
            / "features/ham10000/finetuned_vit_last2_lr5e6_plus_s4_swin_beit/"
        ),
    ),
    (
        "artifacts/features/ham10000/finetuned_vit_last1_lr5e6_plus_s4_swin_beit/",
        str(
            drive_artifact_root
            / "features/ham10000/finetuned_vit_last1_lr5e6_plus_s4_swin_beit/"
        ),
    ),
]
for source, dest in sync_pairs:
    sync_directory(source, dest)

sync_matching_dirs("*e3e*", "artifacts/runs/", str(drive_artifact_root / "runs/"))
sync_matching_files(
    "*finetuned_vit_last*",
    "artifacts/report_assets/tables/",
    str(drive_artifact_root / "report_assets/tables/"),
)
sync_matching_files(
    "*finetuned_vit_last*",
    "artifacts/report_assets/figures/",
    str(drive_artifact_root / "report_assets/figures/"),
)

print(f"E3e artifact sync complete: {drive_artifact_root}")